## Drill Passerone MMM 2026: lammps optimizations of the structures computed with DFT

In [ ]:
import subprocess

def run(cmd):
    process = subprocess.Popen(
        cmd, shell=True, executable="/bin/bash",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT
    )
    for line in process.stdout:
        print(line.decode(), end="")
    process.wait()
    return process.returncode

In [ ]:
run("source /opt/conda/etc/profile.d/conda.sh && conda activate lmp-reaxff && which lmp")


In [ ]:
import numpy as np
import scipy
from scipy.special import sph_harm
from glob import glob
from ase.io import read,write
from ase import neighborlist
import matplotlib.pyplot as plt

import numpy as np
from ase.visualize import view
import matplotlib.pyplot as plt
import nglview as nv


import ase
import ase.io
import ase.lattice.cubic
import ase.md
from ase.md.nvtberendsen import NVTBerendsen
from ase.units import fs, kB
from ase.calculators.lammpsrun import LAMMPS


In [ ]:
from ase.io import read
from ase.io.lammpsdata import write_lammps_data
import numpy as np

In [ ]:
def view_structure(system):
    t = nv.ASEStructure(system)
    w = nv.NGLWidget(t, gui=True)
    
    w.clear_representations()
    w.center()
    
    symbols = system.get_chemical_symbols()
    cu_idx  = [i for i, s in enumerate(symbols) if s == 'Cu']
    ads_idx = [i for i, s in enumerate(symbols) if s != 'Cu']
    
    print(f"Cu: {len(cu_idx)} atoms, adsorbate: {ads_idx}")
    
    w.add_representation('spacefill',      selection=cu_idx,  color='#C07020', radius=1.28, opacity=0.7)
    w.add_representation('licorice', selection=ads_idx, radius=0.4)
    w.add_representation('spacefill', selection=ads_idx, radius=0.6)
    w.add_representation('label',          selection=ads_idx, label_type='atomindex', color='black')
    
    return w


def view_trajectory(trajectory):
    t2 = nv.ASETrajectory(trajectory)
    w2 = nv.NGLWidget(t2, gui=True)
    
    w2.clear_representations()
    
    symbols = trajectory[0].get_chemical_symbols()
    cu_idx  = [i for i, s in enumerate(symbols) if s == 'Cu']
    ads_idx = [i for i, s in enumerate(symbols) if s != 'Cu']
    
    print(f"Cu: {len(cu_idx)} atoms, adsorbate: {ads_idx}")
    
    w2.add_representation('spacefill',      selection=cu_idx,  color='#C07020', radius=1.28, opacity=0.7)
    w2.add_representation('licorice', selection=ads_idx, radius=0.4)   
    w2.add_representation('spacefill', selection=ads_idx, radius=0.6)

    w2.add_representation('label',          selection=ads_idx, label_type='atomindex', color='black')
    
    return w2

## Optimizing the structures from DFT with reaxff, and creating a larger slab for this classical calculation.

In [82]:
def add_layers_below(cu_slab, n_add):
    import numpy as np
    from ase import Atoms

    pos    = cu_slab.get_positions()
    cell   = cu_slab.get_cell()
    z_vals = sorted(set(round(z, 3) for z in pos[:, 2]))

    interlayer = round(z_vals[1] - z_vals[0], 4)

    bot_pos = pos[np.abs(pos[:, 2] - z_vals[0]) < 0.1]
    dists   = []
    for i in range(len(bot_pos)):
        for j in range(i+1, len(bot_pos)):
            d = np.linalg.norm(bot_pos[i,:2] - bot_pos[j,:2])
            if d > 0.1:
                dists.append(d)
    nn_dist = min(dists)
    a = nn_dist * np.sqrt(2)
    print(f"  Inferred lattice constant : a = {a:.4f} Å")
    print(f"  Interlayer spacing        : {interlayer:.4f} Å")

    xy_A = pos[np.abs(pos[:, 2] - z_vals[0]) < 0.1][:, :2]
    xy_B = pos[np.abs(pos[:, 2] - z_vals[1]) < 0.1][:, :2]

    z_bottom = z_vals[0]
    new_pos  = []
    for i in range(1, n_add + 1):
        z_new = z_bottom - i * interlayer
        xy    = xy_B if i % 2 == 1 else xy_A
        for x, y in xy:
            new_pos.append([x, y, z_new])

    new_layers = Atoms(
        symbols=['Cu'] * len(new_pos),
        positions=new_pos,
        cell=cell,
        pbc=[True, True, False]
    )

    combined = new_layers + cu_slab
    combined.set_cell(cell)
    combined.set_pbc([True, True, False])

    print(f"  Added {n_add} layers below → total Cu: {len(combined)}")
    return combined

def calc_adsorption_energy(xyz_file, ffield='ffield_CuCHO_Zhu',
                           lmp_cmd='/home/jovyan/.conda/envs/lmp-reaxff/bin/lmp',
                           nx=1, ny=1, n_add=0, n_fixed=1,
                           workdir=None):

    import os
    if workdir is None:
        workdir = os.path.splitext(os.path.basename(xyz_file))[0]

    os.makedirs(workdir, exist_ok=True)
    print(f"Working directory: {workdir}/")
    
    """
    Calculate adsorption energy: E_ads = E(slab+mol) - E(slab) - E(mol)

    Parameters
    ----------
    xyz_file : input structure (optimised slab + molecule)
    ffield   : ReaxFF force field file
    lmp_cmd  : path to LAMMPS binary
    nx, ny   : supercell repetitions in x and y
    n_add    : perfect Cu layers to add BELOW the optimised slab
    n_fixed  : number of bottom layers to freeze (info only)
    workdir  : directory for all input/output files
    """
    from ase.io import read
    from ase.io.lammpsdata import write_lammps_data
    from ase.build import make_supercell
    from ase import Atoms
    import subprocess, os
    import numpy as np

    # ── Setup working directory ───────────────────────────────
    os.makedirs(workdir, exist_ok=True)
    print(f"Working directory: {workdir}/")

    # ── Load and split ────────────────────────────────────────
    full    = read(xyz_file)
    symbols = np.array(full.get_chemical_symbols())
    cu_mask = symbols == 'Cu'

    cu_slab   = full[cu_mask].copy()
    adsorbate = full[~cu_mask].copy()

    cu_z_vals     = sorted(set(round(z, 3) for z in cu_slab.get_positions()[:, 2]))
    interlayer    = round(cu_z_vals[1] - cu_z_vals[0], 4)
    orig_cell     = cu_slab.get_cell()
    n_orig_layers = len(cu_z_vals)

    nz = n_orig_layers

    orig_az   = full.get_cell()[2, 2]
    orig_maxz = cu_slab.get_positions()[:, 2].max()
    vacuum    = orig_az - orig_maxz

    # ── xy supercell ──────────────────────────────────────────
    cu_super = make_supercell(cu_slab, [[nx,0,0],[0,ny,0],[0,0,1]])


    # ── Add perfect layers below ──────────────────────────────
    if n_add > 0:
        print(f"Adding {n_add} perfect Cu layers below...")
        cu_super = add_layers_below(cu_super, n_add)

    # Fix cell z
    new_cell = cu_super.get_cell().copy()
    new_cell[2, 2] = cu_super.get_positions()[:, 2].max() + vacuum
    cu_super.set_cell(new_cell)

    # ── Re-place adsorbate ────────────────────────────────────
    cu_super_z  = sorted(set(round(z, 3) for z in cu_super.get_positions()[:, 2]))
    new_surf_z  = cu_super_z[-1]
    orig_surf_z = cu_z_vals[-1]

    ads_pos = adsorbate.get_positions().copy()
    ads_pos[:, 0] += (nx // 2) * orig_cell[0, 0]
    ads_pos[:, 1] += (ny // 2) * orig_cell[1, 1]
    ads_pos[:, 2] += new_surf_z - orig_surf_z
    adsorbate.set_positions(ads_pos)

    # ── Assemble three systems ────────────────────────────────
    full_exp = cu_super + adsorbate
    full_exp.set_cell(cu_super.get_cell())
    full_exp.set_pbc([True, True, False])
    full_exp.wrap()

    slab_exp = cu_super.copy()

    mol_exp = adsorbate.copy()
    mol_exp.set_cell(cu_super.get_cell())
    mol_exp.set_pbc([True, True, False])

    print(f"\nSystem  : {xyz_file}")
    print(f"Slab    : {len(cu_super)} Cu atoms  "
          f"({nx}x{ny}x{nz} layers + {n_add} perfect below, fixed: {n_fixed})")
    print(f"Mol     : {len(adsorbate)} atoms  {list(adsorbate.get_chemical_symbols())}")
    print(f"Cell    : {full_exp.cell[0,0]:.4f} x {full_exp.cell[1,1]:.4f} "
          f"x {full_exp.cell[2,2]:.4f} Å")
    print()

    def write_data(atoms, name):
        atoms = atoms.copy()
        atoms.set_initial_charges([0.0] * len(atoms))
        path    = f'{workdir}/{name}.lammps'
        present = set(atoms.get_chemical_symbols())
        so      = [e for e in ['C', 'H', 'O', 'Cu'] if e in present]
        write_lammps_data(path, atoms, specorder=so,
                          atom_style='charge', units='real')
        return path, so

    def write_input(name, data_path, specorder):
        masses     = {'C': 12.011, 'H': 1.008, 'O': 15.999, 'Cu': 63.546}
        mass_lines = '\n'.join(f'mass {i+1} {masses[e]}'
                               for i, e in enumerate(specorder))
        pc_elem    = ' '.join(specorder)
        out_xyz    = f'{workdir}/{name}_reaxff.xyz'

        script = f"""units           real
atom_style      charge
boundary        p p f

read_data       {data_path}

{mass_lines}

pair_style      reaxff NULL checkqeq yes
pair_coeff      * * {ffield} {pc_elem}

fix             qeq all qeq/reaxff 1 0.0 10.0 1.0e-6 reaxff

thermo          1
thermo_style    custom step pe etotal

min_style       cg
minimize        1.0e-8 1.0e-10 10000 50000

dump            final all extxyz 1 {out_xyz}
dump_modify     final element {' '.join(specorder)} sort id
run             0
undump          final
"""
        path = f'{workdir}/{name}.in'
        with open(path, 'w') as f:
            f.write(script)
        return path, out_xyz

    def parse_energy(log_path):
        kcal_to_eV = 1.0 / 23.0609
        with open(log_path) as f:
            lines = f.readlines()
        pe_kcal    = None
        header_idx = None
        for i, line in enumerate(lines):
            if line.strip().startswith('Step') and 'PotEng' in line:
                header_idx = i
        if header_idx is None:
            raise RuntimeError(f"Could not find thermo header in {log_path}")
        for line in lines[header_idx+1:]:
            try:
                vals = line.split()
                if len(vals) >= 2 and vals[0].lstrip('-').isdigit():
                    pe_kcal = float(vals[1])
            except:
                break
        if pe_kcal is None:
            raise RuntimeError(f"Could not parse energy from {log_path}")
        return pe_kcal * kcal_to_eV

    def run_lammps(atoms, name):
        data_path, specorder = write_data(atoms, name)
        in_path, out_xyz     = write_input(name, data_path, specorder)
        log_path             = f'{workdir}/{name}.log'

        cmd    = f'{lmp_cmd} -in {in_path} -log {log_path}'
        result = subprocess.run(cmd, shell=True, capture_output=True, text=True)

        if result.returncode != 0:
            print(f"  LAMMPS stderr:\n{result.stderr[-2000:]}")
            raise RuntimeError(f"LAMMPS failed for {name}, check {log_path}")

        e = parse_energy(log_path)
        print(f"  E({name:12s}) = {e:.6f} eV  ({e*23.0609:.4f} kcal/mol)")
        print(f"  Written: {out_xyz}")
        return e

    # ── Run three calculations ────────────────────────────────
    e_full = run_lammps(full_exp, 'slab_mol')
    e_slab = run_lammps(slab_exp, 'slab')
    e_mol  = run_lammps(mol_exp,  'mol')

    e_ads = e_full - e_slab - e_mol

    print()
    print(f"  E_ads = E(slab+mol) - E(slab) - E(mol)")
    print(f"        = {e_full:.6f} - {e_slab:.6f} - {e_mol:.6f}")
    print(f"        = {e_ads:.6f} eV  ({e_ads*23.0609:.4f} kcal/mol)")

    return {'e_ads': e_ads, 'e_full': e_full, 'e_slab': e_slab,
            'e_mol': e_mol, 'units': 'eV'}




Working directory: eads_CO/
Working directory: eads_CO/
Adding 2 perfect Cu layers below...
  Inferred lattice constant : a = 3.6100 Å
  Interlayer spacing        : 1.7560 Å
  Added 2 layers below → total Cu: 864

System  : slab_CO-gopt.xyz
Slab    : 864 Cu atoms  (3x3x21 layers + 2 perfect below, fixed: 1)
Mol     : 2 atoms  ['C', 'O']
Cell    : 30.6330 x 30.6330 x 35.4150 Å

  E(slab_mol    ) = -2904.141599 eV  (-66972.1190 kcal/mol)
  Written: eads_CO/slab_mol_reaxff.xyz
  E(slab        ) = -2892.065661 eV  (-66693.6370 kcal/mol)
  Written: eads_CO/slab_reaxff.xyz
  E(mol         ) = -11.024731 eV  (-254.2402 kcal/mol)
  Written: eads_CO/mol_reaxff.xyz

  E_ads = E(slab+mol) - E(slab) - E(mol)
        = -2904.141599 - -2892.065661 - -11.024731
        = -1.051207 eV  (-24.2418 kcal/mol)

Adsorption energy CO : -1.0512 eV


## Use of the above functions to compute the adsorption energies. You should have in this directory all the slab_molecule-gopt.xyz  structures you created in CP2K Lab.

### Here an example with CO adsorption. Repeat with the 2 CO adsorption, and with the C2O2 dimer adsorption

In [ ]:

results = calc_adsorption_energy(
    'slab_CO-gopt.xyz',
    ffield='ffield_CuCHO_Zhu',
    nx=3, ny=3,
    n_add=2,
    n_fixed=1,
    workdir='eads_CO',
)
print(f"\nAdsorption energy CO : {results['e_ads']:.4f} eV")

In [ ]:
atoms = read("slab_CO-gopt/slab_mol_reaxff.xyz")
view_structure_mixed(atoms)